# U08_P — Nucleotide Analysis — Solutions
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 8:** Predictive Methods using Nucleotide Sequence
**Type:** Practice Solutions notebook · **Time:** 50 min · **Difficulty:** ★★☆
**Goes with:** Lectures L21–L22, Practical S9
**Before this:** U08_C1

In [1]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U08_P"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

Python 3.11.18 | Biopython 1.88


### Ex 8.1 ★☆☆ — Count stop codons in Frame +1 of HBB mRNA
**Biology:** Non-coding frames typically contain numerous stop codons.  
**Task:** Translate `hbb_human_mrna.fasta` in Frame +1 (starting at index 0) and count the number of stop codons (`'*'`). Store the count in `frame1_stops`.  
**Expected:** `frame1_stops` as an integer > 5.

In [2]:
from Bio import SeqIO

rec = SeqIO.read(DATA_DIR / "sequences" / "hbb_human_mrna.fasta", "fasta")
dna = rec.seq
trans_f1 = dna.translate()
frame1_stops = trans_f1.count("*")

/Users/prateekpaul/miniconda3/envs/cheminf/lib/python3.11/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [3]:
# Check — do not edit
assert frame1_stops > 5, f"Expected > 5 stop codons in non-coding frame, got {frame1_stops}"
print("Ex 8.1 passed")

Ex 8.1 passed


### Ex 8.2 ★☆☆ — Identify canonical stop codons
**Biology:** Standard nuclear genetic code uses three stop codons: TAA (ochre), TAG (amber), and TGA (opal).  
**Task:** Create a set `stop_codons` containing the three canonical DNA stop codon strings. Verify whether `'TGA'` is in `stop_codons`.  
**Expected:** `stop_codons` is a set with 3 elements.

In [4]:
stop_codons = {"TAA", "TAG", "TGA"}

In [5]:
# Check — do not edit
assert len(stop_codons) == 3
assert "TAA" in stop_codons and "TAG" in stop_codons and "TGA" in stop_codons
print("Ex 8.2 passed")

Ex 8.2 passed


### Ex 8.3 ★★☆ — Find the primary coding ORF in human insulin mRNA
**Biology:** Human preproinsulin (INS) is encoded by NM_000207.3.  
**Task:** Scan `ins_human_mrna.fasta` for the longest ORF starting with `'ATG'` and ending with a canonical stop codon. Store its amino acid length in `ins_orf_aa_len`.  
**Expected:** `ins_orf_aa_len` == 110.

In [6]:
ins_dna = str(SeqIO.read(DATA_DIR / "sequences" / "ins_human_mrna.fasta", "fasta").seq).upper()

longest_aa = 0
for frame in range(3):
    for i in range(frame, len(ins_dna) - 2, 3):
        if ins_dna[i:i+3] == "ATG":
            for j in range(i + 3, len(ins_dna) - 2, 3):
                if ins_dna[j:j+3] in stop_codons:
                    aa_len = (j + 3 - i - 3) // 3
                    if aa_len > longest_aa:
                        longest_aa = aa_len
                    break

ins_orf_aa_len = longest_aa

In [7]:
# Check — do not edit
assert ins_orf_aa_len == 110, f"Expected 110 amino acids for preproinsulin ORF, got {ins_orf_aa_len}"
print("Ex 8.3 passed")

Ex 8.3 passed


### Ex 8.4 ★★☆ — Simulate beta-thalassemia codon 39 nonsense mutation
**Biology:** A common Mediterranean beta-thalassemia mutation mutates codon 39 (CAG, Gln) to UAG (stop), truncating beta-globin.  
**Task:** Take the wild-type HBB coding sequence `hbb_cds`. Mutate codon 39 from `'CAG'` to `'TAG'`. Translate to stop and store the resulting truncated peptide length in `trunc_len`.  
**Expected:** `trunc_len` == 38 amino acids.

In [8]:
hbb_cds = str(SeqIO.read(DATA_DIR / "sequences" / "hbb_human_cds.fasta", "fasta").seq)

c39_start = 38 * 3
mutated_cds = hbb_cds[:c39_start] + "TAG" + hbb_cds[c39_start + 3:]

from Bio.Seq import Seq
trunc_peptide = Seq(mutated_cds).translate(to_stop=True)
trunc_len = len(trunc_peptide)

In [9]:
# Check — do not edit
assert trunc_len == 38, f"Expected 38 aa truncated peptide, got {trunc_len}"
print("Ex 8.4 passed")

Ex 8.4 passed


### Ex 8.5 ★★☆ — Calculate overall codon usage in human HBB CDS
**Biology:** Quantifying codon frequency across a coding sequence.  
**Task:** Iterate across `hbb_cds` in codons of 3. Store the total number of codons in `total_codons`, and count how many times `'CTG'` occurs in `ctg_count`.  
**Expected:** `total_codons` = 148 (147 amino acids + stop), `ctg_count` > 5.

In [10]:
codons = [hbb_cds[i:i+3] for i in range(0, len(hbb_cds) - 2, 3)]
total_codons = len(codons)
ctg_count = codons.count("CTG")

In [11]:
# Check — do not edit
assert total_codons == 148, f"Expected 148 codons, got {total_codons}"
assert ctg_count >= 5, f"Expected >= 5 CTG codons, got {ctg_count}"
print("Ex 8.5 passed")

Ex 8.5 passed


### Ex 8.6 ★★★ — Calculate GC skew along sliding windows
**Biology:** GC skew $(G - C) / (G + C)$ changes sign at bacterial replication origins and reveals transcription strand asymmetry.  
**Task:** Write a function `calc_gc_skew(dna_seq)` returning $(G - C) / (G + C)$ as a float rounded to 3 decimal places (return 0.0 if $G+C=0$). Compute the skew for `hbb_cds` in `cds_skew`.  
**Expected:** `cds_skew` as a float between -0.5 and +0.5.

In [12]:
def calc_gc_skew(dna_seq):
    g = dna_seq.count("G")
    c = dna_seq.count("C")
    if g + c == 0:
        return 0.0
    return round((g - c) / (g + c), 3)

cds_skew = calc_gc_skew(hbb_cds)

In [13]:
# Check — do not edit
assert isinstance(cds_skew, float)
assert -0.5 <= cds_skew <= 0.5, f"Unexpected GC skew: {cds_skew}"
print("Ex 8.6 passed")

Ex 8.6 passed


## 5. Reflection Questions
1. Why does a random sequence of DNA contain stop codons approximately every 21 codons on average, and why does this make long ORFs good candidate protein-coding genes?
2. What are the major differences between gene prediction in prokaryotes (dense, continuous ORFs) and eukaryotes (interrupted by introns, alternative splicing)?